# AI/ML AWS AASTU Cohort — Week 2 Lab: Mini Semantic Search

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/)

## 🎯 Objective
Build a production-grade **Mini Semantic Search Engine** from scratch using:
- **Pandas** for structured FAQ loading and data manipulation
- Custom NLP functions for text cleaning and overlapping text chunking
- **Sentence-Transformers** for generating dense numerical vector embeddings
- **ChromaDB** as the AI-native vector database for indexing and querying
- Retrieve and evaluate the **Top 3 semantic matches** for any natural language inquiry

> **Architecture Note**: This notebook reuses all modular production functions defined in the [`src/`](../src/) package (`data_loader`, `preprocessor`, `vector_store`, `search_engine`).

### Step 0: Install Dependencies (Optional)
Uncomment and run if executing in Google Colab or a fresh virtual environment.

In [1]:
# %pip install -q pandas numpy chromadb sentence-transformers tabulate

### Step 1: Import Reusable Modular Functions from `src`

In [2]:
import os
import sys

# Ensure project root is in sys.path so we can import from src
for path in [os.path.abspath("."), os.path.abspath(".."), os.path.abspath("../..")]:
    if os.path.isdir(os.path.join(path, "src")) and path not in sys.path:
        sys.path.insert(0, path)
        break

# Import modular functions directly from the src package
from src import (
    load_faqs,
    clean_text,
    chunk_text,
    prepare_faq_documents,
    initialize_chroma_collection,
    index_faqs_to_chroma,
    query_faqs,
    print_search_results,
    search_and_display,
)

print("✅ Successfully imported modular functions from 'src'!")

✅ Successfully imported modular functions from 'src'!


## 📂 Step 2: Load FAQ Dataset with Pandas

We use `load_faqs()` from `src.data_loader` to ingest and validate the FAQ dataset.

In [3]:
# Load FAQ dataset
dataset_path = os.path.join("data", "faqs.csv")
df_faqs = load_faqs(dataset_path)

print(f"✅ Successfully loaded {len(df_faqs)} FAQs with columns: {list(df_faqs.columns)}")
df_faqs.head(5)

✅ Successfully loaded 20 FAQs with columns: ['id', 'question', 'answer', 'category']


,id,question,answer,category
0,1,What is Amazon SageMaker and what is it used for?,Amazon SageMaker is a fully managed cloud mach...,Machine Learning
1,2,What is Amazon Bedrock and how does it support...,Amazon Bedrock is a fully managed AWS service ...,Generative AI
2,3,What is the difference between Amazon S3 and A...,Amazon S3 (Simple Storage Service) is an objec...,Storage
3,4,What is AWS Lambda and what are its key limita...,AWS Lambda is a serverless compute service tha...,Serverless Compute
4,5,What is a vector embedding in machine learning...,A vector embedding is a high-dimensional dense...,Vector Search & Embeddings


## 🧹 Step 3: Text Cleaning and Chunking

We use `clean_text()` and `chunk_text()` from `src.preprocessor` with a sliding window to preserve context across chunk boundaries.

In [4]:
# Demonstrate sliding-window chunking on a single FAQ
sample_answer = df_faqs.iloc[0]["answer"]
sample_chunks = chunk_text(sample_answer, chunk_size=120, chunk_overlap=30)

print(f"Original text length: {len(sample_answer)} characters")
print(f"Generated {len(sample_chunks)} overlapping chunks:")
for i, ch in enumerate(sample_chunks):
    print(f"  [Chunk {i+1}]: {ch}")

Original text length: 277 characters
Generated 3 overlapping chunks:
  [Chunk 1]: Amazon SageMaker is a fully managed cloud machine learning platform provided by AWS. It enables developers and data scie
  [Chunk 2]: ables developers and data scientists to build, train, tune, and deploy machine learning models at scale with integrated
  [Chunk 3]: dels at scale with integrated tools like SageMaker Studio, Model Registry, and managed endpoints.


### Prepare All FAQ Documents for Vector Indexing

In [5]:
# Transform entire dataset into chunked documents with metadata
documents, metadatas, ids = prepare_faq_documents(df_faqs, chunk_size=200, chunk_overlap=40)

print(f"✅ Total chunks prepared for vector indexing: {len(documents)}")
print(f"Sample Doc ID : {ids[0]}")
print(f"Sample Chunk  : {documents[0]}")
print(f"Sample Meta   : {metadatas[0]}")

✅ Total chunks prepared for vector indexing: 55
Sample Doc ID : faq_1_chunk_0
Sample Chunk  : Question: What is Amazon SageMaker and what is it used for? Answer: Amazon SageMaker is a fully managed cloud machine learning platform provided by AWS. It enables developers and data scientists to bu
Sample Meta   : {'faq_id': 1, 'question': 'What is Amazon SageMaker and what is it used for?', 'category': 'Machine Learning', 'chunk_index': 0, 'total_chunks': 2}


## 🧠 Step 4: Embed and Store in ChromaDB

We use `initialize_chroma_collection()` and `index_faqs_to_chroma()` from `src.vector_store` to initialize the vector store and index all chunks.

In [ ]:
# Initialize ChromaDB collection
collection = initialize_chroma_collection(
    collection_name="aws_week_2_faqs",
    persist_directory="./chroma_db",
)

# Index all chunks
indexed_count = index_faqs_to_chroma(collection, documents, metadatas, ids)
print(f"✅ Successfully indexed {indexed_count} chunks into collection '{collection.name}'!")

c:\Users\hp\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 🔍 Step 5: Query Semantic Search & Print Top 3 Matches

We use `search_and_display()` from `src.search_engine` to query the vector collection and display the Top 3 results.

### 🔹 Query 1: Cloud ML Model Deployment
Matches **Amazon SageMaker** through conceptual similarity.

In [ ]:
search_and_display(collection, "How can I deploy and train machine learning models on AWS?", top_k=3);

### 🔹 Query 2: Serverless Operational Constraints
Matches **AWS Lambda** based on the 15-minute execution limit.

In [ ]:
search_and_display(collection, "What is the timeout limit for serverless functions?", top_k=3);

### 🔹 Query 3: Vector Similarity vs Keyword Matching
Retrieves conceptual comparisons between dense embeddings and keyword matching.

In [ ]:
search_and_display(collection, "How does vector similarity search work compared to keyword matching?", top_k=3);

### 🔹 Query 4: GPU Hardware for Machine Learning
Matches **EC2 accelerated computing families** (G-series and P-series).

In [ ]:
search_and_display(collection, "Which GPU instance family is optimized for deep learning training?", top_k=3);

## 💡 Interactive Custom Search
Enter any custom query to test the semantic search engine:

In [ ]:
custom_query = "What service provides foundation models via API?"
search_and_display(collection, custom_query, top_k=3);

## 📊 Step 6: Summary & Architectural Benefits

| Benefit | How It Is Implemented |
| :--- | :--- |
| **Single Source of Truth** | All business logic resides in `src/` modules, reused by both `main.py` and this notebook |
| **Context Preservation** | Text is chunked with a 40-character sliding overlap to prevent boundary information loss |
| **ChromaDB Vector Store** | Dense embeddings indexed with cosine distance metric for fast k-NN retrieval |
| **Top-K Ranking** | Queries return top 3 matches ranked by semantic similarity score |